<a href="https://colab.research.google.com/github/laav-marusu/Air-Pollution-Monitoring-System/blob/main/RAG_Step3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
!pip install -q langchain-text-splitters chromadb sentence-transformers

from langchain_text_splitters import RecursiveCharacterTextSplitter
import chromadb
from chromadb.utils import embedding_functions

documents = {
    "ai_overview.txt": """Artificial Intelligence: An Overview

Artificial Intelligence, or AI, is the field of computer science focused on building systems
that can perform tasks which normally require human intelligence. These tasks include
understanding language, recognizing images, making decisions, and learning from experience.

Machine Learning is a subset of AI where systems learn patterns from data instead of following
hardcoded rules. Instead of programming every decision by hand, we feed the system many examples,
and it learns the underlying pattern on its own.

Deep Learning takes Machine Learning further by using neural networks with many layers. These
layered networks can automatically discover complex patterns in large amounts of data, such as
recognizing faces in photos or understanding the meaning of a sentence.

Retrieval-Augmented Generation, or RAG, is a technique that combines a search system with a
language model. Instead of relying only on what a language model memorized during training, RAG
first retrieves relevant, up-to-date information from a document collection, then passes that
information to the model so it can generate a more accurate, grounded answer.
""",
    "embeddings_in_practice.txt": """Embeddings and Vector Search in Practice

Modern search engines increasingly rely on embeddings rather than simple keyword matching.
An embedding model converts a search query into the same vector space as the stored documents,
allowing the system to find results that match the meaning of the query, not just its exact words.

This is especially powerful for customer support systems, where a user might phrase a question very
differently from how it appears in a help article, yet still expect to find the right answer.

Vector databases such as ChromaDB store these embeddings and allow fast similarity search across
thousands or millions of documents at once.
"""
}

splitter = RecursiveCharacterTextSplitter(chunk_size=250, chunk_overlap=40, separators=["\n\n", "\n", ". ", " ", ""])

client = chromadb.Client()
hf_embedder = embedding_functions.SentenceTransformerEmbeddingFunction(model_name="all-MiniLM-L6-v2")
collection = client.get_or_create_collection(name="rag_chunks", embedding_function=hf_embedder)

for source, text in documents.items():
    doc_chunks = splitter.split_text(text)
    collection.add(
        documents=doc_chunks,
        metadatas=[{"source": source, "chunk_index": i} for i in range(len(doc_chunks))],
        ids=[f"{source}_chunk_{i}" for i in range(len(doc_chunks))]
    )

print("Total chunks stored:", collection.count())

Total chunks stored: 13


In [11]:
user_query = "How does RAG improve answers from a language model?"

results = collection.query(
    query_texts=[user_query],
    n_results=3,
    include=["documents", "metadatas", "distances"]
)

for doc, meta, distance in zip(results["documents"][0], results["metadatas"][0], results["distances"][0]):
    print(f"Distance: {distance:.4f} | Source: {meta['source']} | Chunk #{meta['chunk_index']}")
    print(doc)
    print("-" * 60)

Distance: 0.3244 | Source: ai_overview.txt | Chunk #6
Retrieval-Augmented Generation, or RAG, is a technique that combines a search system with a
language model. Instead of relying only on what a language model memorized during training, RAG
------------------------------------------------------------
Distance: 0.6356 | Source: ai_overview.txt | Chunk #7
first retrieves relevant, up-to-date information from a document collection, then passes that
information to the model so it can generate a more accurate, grounded answer.
------------------------------------------------------------
Distance: 0.6794 | Source: embeddings_in_practice.txt | Chunk #2
allowing the system to find results that match the meaning of the query, not just its exact words.
------------------------------------------------------------


In [12]:
def distance_to_similarity(distance):
    return 1 - distance

for doc, meta, distance in zip(results["documents"][0], results["metadatas"][0], results["distances"][0]):
    similarity = distance_to_similarity(distance)
    print(f"Similarity: {similarity:.4f} | Source: {meta['source']} | Chunk #{meta['chunk_index']}")

Similarity: 0.6756 | Source: ai_overview.txt | Chunk #6
Similarity: 0.3644 | Source: ai_overview.txt | Chunk #7
Similarity: 0.3206 | Source: embeddings_in_practice.txt | Chunk #2


In [13]:
def distance_to_similarity(distance):
    return 1 - distance

def retrieve_with_threshold(query, collection, n_results=5, similarity_threshold=0.3):
    results = collection.query(
        query_texts=[query],
        n_results=n_results,
        include=["documents", "metadatas", "distances"]
    )

    filtered = []
    for doc, meta, distance in zip(results["documents"][0], results["metadatas"][0], results["distances"][0]):
        similarity = distance_to_similarity(distance)
        if similarity >= similarity_threshold:
            filtered.append({"text": doc, "metadata": meta, "similarity": similarity})

    return filtered

# A relevant query
relevant_matches = retrieve_with_threshold("What is deep learning?", collection, similarity_threshold=0.3)
print(f"Relevant query -> {len(relevant_matches)} chunk(s) passed the threshold\n")
for m in relevant_matches:
    print(f"Similarity: {m["similarity"]:.4f} | {m["text"][:80]}...")

Relevant query -> 4 chunk(s) passed the threshold

Similarity: 0.6346 | Deep Learning takes Machine Learning further by using neural networks with many ...
Similarity: 0.4946 | Machine Learning is a subset of AI where systems learn patterns from data instea...
Similarity: 0.3875 | Artificial Intelligence, or AI, is the field of computer science focused on buil...
Similarity: 0.3682 | understanding language, recognizing images, making decisions, and learning from ...


In [14]:
# An unrelated query - notice fewer (or zero) chunks pass the threshold
unrelated_matches = retrieve_with_threshold("What is the best recipe for chocolate cake?", collection, similarity_threshold=0.3)
print(f"Unrelated query -> {len(unrelated_matches)} chunk(s) passed the threshold")

if len(unrelated_matches) == 0:
    print("No relevant context found - a real RAG system should say 'I don't know' here instead of guessing.")

Unrelated query -> 0 chunk(s) passed the threshold
No relevant context found - a real RAG system should say 'I don't know' here instead of guessing.


In [15]:
results = collection.query(
    query_texts=["how does search work with meaning instead of keywords"],
    n_results=2,
    where={"source": "embeddings_in_practice.txt"},
    include=["documents", "metadatas", "distances"]
)

for doc, meta, distance in zip(results["documents"][0], results["metadatas"][0], results["distances"][0]):
    print(f"Similarity: {distance_to_similarity(distance):.4f} | Source: {meta['source']}")
    print(doc)
    print("-" * 60)

Similarity: 0.6371 | Source: embeddings_in_practice.txt
allowing the system to find results that match the meaning of the query, not just its exact words.
------------------------------------------------------------
Similarity: 0.6114 | Source: embeddings_in_practice.txt
Modern search engines increasingly rely on embeddings rather than simple keyword matching.
An embedding model converts a search query into the same vector space as the stored documents,
------------------------------------------------------------


In [16]:
def retrieve_context(query, collection, n_results=5, similarity_threshold=0.25, source_filter=None):
    query_kwargs = {
        "query_texts": [query],
        "n_results": n_results,
        "include": ["documents", "metadatas", "distances"]
    }
    if source_filter:
        query_kwargs["where"] = {"source": source_filter}

    results = collection.query(**query_kwargs)

    context_chunks = []
    for doc, meta, distance in zip(results["documents"][0], results["metadatas"][0], results["distances"][0]):
        similarity = distance_to_similarity(distance)
        if similarity >= similarity_threshold:
            context_chunks.append({
                "text": doc,
                "source": meta["source"],
                "chunk_index": meta["chunk_index"],
                "similarity": round(similarity, 4)
            })

    return context_chunks

context = retrieve_context("What is machine learning?", collection)
for c in context:
    print(c)

{'text': 'Machine Learning is a subset of AI where systems learn patterns from data instead of following\nhardcoded rules. Instead of programming every decision by hand, we feed the system many examples,\nand it learns the underlying pattern on its own.', 'source': 'ai_overview.txt', 'chunk_index': 3, 'similarity': 0.7744}
{'text': 'Deep Learning takes Machine Learning further by using neural networks with many layers. These\nlayered networks can automatically discover complex patterns in large amounts of data, such as', 'source': 'ai_overview.txt', 'chunk_index': 4, 'similarity': 0.5253}
{'text': 'Artificial Intelligence, or AI, is the field of computer science focused on building systems\nthat can perform tasks which normally require human intelligence. These tasks include', 'source': 'ai_overview.txt', 'chunk_index': 1, 'similarity': 0.4885}
{'text': 'Artificial Intelligence: An Overview', 'source': 'ai_overview.txt', 'chunk_index': 0, 'similarity': 0.3746}
{'text': 'understanding l

In [17]:
def format_context_for_llm(query, context_chunks):
    if not context_chunks:
        return f"Question: {query}\n\nNo relevant context was found in the knowledge base."

    context_text = "\n\n".join(
        f"[Source: {c['source']}, chunk {c['chunk_index']}]\n{c['text']}"
        for c in context_chunks
    )

    prompt = f"""Answer the question using ONLY the context below. If the answer isn't in the context, say you don't know.

Context:
{context_text}

Question: {query}

Answer:"""
    return prompt

context = retrieve_context("How does RAG combine search with a language model?", collection)
final_prompt = format_context_for_llm("How does RAG combine search with a language model?", context)

print(final_prompt)

Answer the question using ONLY the context below. If the answer isn't in the context, say you don't know.

Context:
[Source: ai_overview.txt, chunk 6]
Retrieval-Augmented Generation, or RAG, is a technique that combines a search system with a
language model. Instead of relying only on what a language model memorized during training, RAG

[Source: embeddings_in_practice.txt, chunk 2]
allowing the system to find results that match the meaning of the query, not just its exact words.

[Source: embeddings_in_practice.txt, chunk 1]
Modern search engines increasingly rely on embeddings rather than simple keyword matching.
An embedding model converts a search query into the same vector space as the stored documents,

[Source: embeddings_in_practice.txt, chunk 0]
Embeddings and Vector Search in Practice

[Source: ai_overview.txt, chunk 7]
first retrieves relevant, up-to-date information from a document collection, then passes that
information to the model so it can generate a more accurate, gro

In [18]:
test_queries = [
    "Why do companies use vector databases?",
    "What's the difference between AI and machine learning?",
    "How do chatbots understand differently worded questions?",
]

for q in test_queries:
    print(f"Query: {q}")
    matches = retrieve_context(q, collection, n_results=2)
    for m in matches:
        print(f"   [{m['similarity']}] ({m['source']}) {m['text'][:90]}...")
    print()

Query: Why do companies use vector databases?
   [0.4543] (embeddings_in_practice.txt) Embeddings and Vector Search in Practice...
   [0.4013] (embeddings_in_practice.txt) Vector databases such as ChromaDB store these embeddings and allow fast similarity search ...

Query: What's the difference between AI and machine learning?
   [0.6668] (ai_overview.txt) Artificial Intelligence, or AI, is the field of computer science focused on building syste...
   [0.6572] (ai_overview.txt) Machine Learning is a subset of AI where systems learn patterns from data instead of follo...

Query: How do chatbots understand differently worded questions?
   [0.4263] (embeddings_in_practice.txt) allowing the system to find results that match the meaning of the query, not just its exac...
   [0.3312] (embeddings_in_practice.txt) This is especially powerful for customer support systems, where a user might phrase a ques...

